<a href="https://colab.research.google.com/github/rajaveni08-rgb/Trendpulse_Rajaveni/blob/main/Mini_Projects_Rajaveni_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import json
import os
from datetime import datetime

def collect_trends():
    # Ensure data directory exists
    os.makedirs("data", exist_ok=True)

    # Mock data structure matching a typical trending payload
    # Replace this block with your actual web-scraping or API code
    mock_raw_data = [
        {"title": "Automotive AI Agents", "category": "Tech", "volume": 120000, "region": "US"},
        {"title": "Electric Trucks", "category": "Automotive", "volume": 85000, "region": "US"},
        {"title": "LLM Edge Computing", "category": "Tech", "volume": 95000, "region": "EU"},
        {"title": "Next-gen Batteries", "category": "Automotive", "volume": 64000, "region": "APAC"},
        {"title": "Autonomous Delivery", "category": "Logistics", "volume": 43000, "region": "US"}
    ]

    # Generate filepath dynamically based on current date
    date_str = datetime.now().strftime("%Y%m%d")
    filepath = f"data/trends_{date_str}.json"

    with open(filepath, "w") as f:
        json.dump(mock_raw_data, f, indent=4)

    print(f"[Task 1 Complete] Raw trend data collected and written to: {filepath}")

if __name__ == "__main__":
    collect_trends()


[Task 1 Complete] Raw trend data collected and written to: data/trends_20260929.json


In [11]:
import os
import glob
import pandas as pd

def process_data():
    # Locate the latest JSON file in the data folder
    json_files = glob.glob("data/trends_*.json")
    if not json_files:
        raise FileNotFoundError("No raw trend JSON files found in data/")

    latest_file = max(json_files, key=os.path.getctime)
    print(f"Processing latest raw file: {latest_file}")

    # Load into data frame
    df = pd.read_json(latest_file)

    # Cleaning steps (Handling missing values, standardizing column formatting, filtering)
    df.dropna(subset=['title'], inplace=True)
    df['title'] = df['title'].str.strip()
    df['category'] = df['category'].str.capitalize()

    # Export to clean CSV
    output_path = "data/trends_clean.csv"
    df.to_csv(output_path, index=False)
    print(f"[Task 2 Complete] Sanitized data exported to: {output_path}")

if __name__ == "__main__":
    process_data()


Processing latest raw file: data/trends_20260929.json
[Task 2 Complete] Sanitized data exported to: data/trends_clean.csv


In [12]:
import pandas as pd

def analyze_data():
    input_path = "data/trends_clean.csv"
    if not pd.io.common.file_exists(input_path):
        raise FileNotFoundError(f"Missing processed data file: {input_path}")

    df = pd.read_csv(input_path)

    # Perform Analysis logic (Example: Calculating global share and engagement index metrics)
    total_volume = df['volume'].sum()
    df['volume_share_pct'] = (df['volume'] / total_volume) * 100

    # Engineering a pseudo-momentum metrics matrix
    # Higher volume fields in specific key markets score higher scales
    df['pulse_score'] = df['volume'] * df['region'].apply(lambda r: 1.2 if r == 'US' else 1.0)

    # Sort results by the calculated intensity profile
    df = df.sort_values(by='pulse_score', ascending=False)

    output_path = "data/trends_analysed.csv"
    df.to_csv(output_path, index=False)
    print(f"[Task 3 Complete] Metrics analyzed. Dataset saved to: {output_path}")

if __name__ == "__main__":
    analyze_data()


[Task 3 Complete] Metrics analyzed. Dataset saved to: data/trends_analysed.csv


In [13]:
import os
import pandas as pd
import matplotlib.pyplot as plt

def generate_visualizations():
    os.makedirs("outputs", exist_ok=True)
    df = pd.read_csv("data/trends_analysed.csv")

    plt.style.use('ggplot') # Modern look for output files

    # 1. Chart 1: Bar chart of Top Stories
    plt.figure(figsize=(6, 4))
    plt.barh(df['title'].head(5), df['volume'].head(5), color='royalblue')
    plt.gca().invert_yaxis()
    plt.title('Top Trending Stories by Volume')
    plt.tight_layout()
    plt.savefig('outputs/chart1_top_stories.png')
    plt.close()

    # 2. Chart 2: Category Breakdown
    plt.figure(figsize=(6, 4))
    cat_data = df.groupby('category')['volume'].sum()
    cat_data.plot(kind='bar', color='seagreen')
    plt.title('Total Volume by Industry Category')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('outputs/chart2_categories.png')
    plt.close()

    # 3. Chart 3: Scatter Plot (Volume Share vs Pulse Score)
    plt.figure(figsize=(6, 4))
    plt.scatter(df['volume_share_pct'], df['pulse_score'], s=150, color='crimson', alpha=0.7)
    for i, txt in enumerate(df['title']):
        plt.annotate(txt, (df['volume_share_pct'].iloc[i], df['pulse_score'].iloc[i]), fontsize=8)
    plt.title('Trend Momentum Scatter Matrix')
    plt.tight_layout()
    plt.savefig('outputs/chart3_scatter.png')
    plt.close()

    # 4. Bonus: Dashboard Generation (Combines subplots cleanly into a single file grid)
    fig, axs = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle('TrendPulse Analysis System Executive Dashboard', fontsize=16, fontweight='bold')

    # Subplot 1
    axs[0, 0].barh(df['title'].head(5), df['volume'].head(5), color='royalblue')
    axs[0, 0].invert_yaxis()
    axs[0, 0].set_title('Top Volume Rankings')

    # Subplot 2
    cat_data.plot(kind='bar', ax=axs[0, 1], color='seagreen')
    axs[0, 1].set_title('Categories Share')

    # Subplot 3
    axs[1, 0].scatter(df['volume_share_pct'], df['pulse_score'], s=100, color='crimson')
    axs[1, 0].set_title('Momentum Cluster Map')

    # Subplot 4: Data table visualization summarizing findings
    axs[1, 1].axis('off')
    table_data = df[['title', 'region', 'pulse_score']].head(4)
    table = axs[1, 1].table(cellText=table_data.values, colLabels=table_data.columns, loc='center', cellLoc='center')
    table.scale(1, 1.5)
    axs[1, 1].set_title('Summary Metrics View')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig('outputs/dashboard.png', dpi=150)
    plt.close()

    print("[Task 4 Complete] Generated chart1, chart2, chart3, and the bonus dashboard inside outputs/.")

if __name__ == "__main__":
    generate_visualizations()


[Task 4 Complete] Generated chart1, chart2, chart3, and the bonus dashboard inside outputs/.
